# Ejercicios — Regresión Ridge

- 📘 Explicación: [`../explained/4_regresion_lineal_regularizacion_ridge.ipynb`](../explained/4_regresion_lineal_regularizacion_ridge.ipynb)
- 📓 Notebook de clase: [`../raw/4_regresion_lineal_regularizacion_ridge.ipynb`](../raw/4_regresion_lineal_regularizacion_ridge.ipynb)

> Los enunciados están tal cual los dio la cátedra. Las pistas (*centros*) están al final de la notebook explicada.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../../datasets")


---

### Ejercicio 1 — Subset selection

**Ejercicio - Investigar:** ¿Qué es *subset selection*?


**Mi respuesta:**

**Subset selection** (📕 ESL §3.3) es la forma más directa de simplificar un modelo lineal: en vez de usar los $p$ predictores, **elegir un subconjunto de ellos** y ajustar cuadrados mínimos solo con esos. Los coeficientes de las variables que quedan afuera valen exactamente $0$.

- **Best subset selection.** Para cada tamaño $k \in \{0, 1, \dots, p\}$ se busca el subconjunto de $k$ variables con menor RSS:
  $$\min_{\beta}\ \|\mathbf{y} - \mathbf{X}\beta\|^2 \quad \text{sujeto a} \quad \#\{j : \beta_j \neq 0\} \le k$$
  Eso da una secuencia de $p+1$ modelos candidatos. El $k$ final **no** se elige por RSS (el RSS siempre baja al agregar variables, así que ganaría siempre $k = p$), sino por error estimado fuera de muestra: validación cruzada, $C_p$, AIC o BIC.
- **Costo.** Hay $2^p$ subconjuntos. Con el algoritmo *leaps and bounds* (Furnival & Wilson, 1974) se puede hacer hasta $p \approx 30$–$40$; más allá es inviable.
- **Alternativas greedy** para $p$ grande:
  - *Forward stepwise*: arranca con el intercepto y agrega de a una la variable que más baja el RSS.
  - *Backward stepwise*: arranca con todas y saca de a una la que menos aporta (necesita $N > p$).
  - *Forward stagewise*: todavía más conservador; en cada paso mueve un poquito el coeficiente de la variable más correlacionada con el residuo.

**Relación con Ridge y Lasso.** Subset selection es la penalización $L_0$: $\lambda \sum_j \mathbb{1}[\beta_j \neq 0]$, el caso $q \to 0$ de la familia $\lambda\sum_j|\beta_j|^q$. Produce modelos interpretables (tira variables), pero es un proceso **discreto**: una variable entra o sale de golpe, así que un cambio chico en los datos puede cambiar mucho el modelo elegido. Eso le da **alta varianza**, y por eso a menudo no reduce el error de predicción tanto como uno esperaría. Ridge es la alternativa **continua**: en vez de tirar variables, encoge todos los coeficientes de forma suave. Lasso ($q = 1$) queda en el medio: es continuo y convexo como Ridge, pero sí manda coeficientes a cero exacto como subset selection.


---

### Ejercicio 2 — Ridge sobre Prostate Cancer

Utilizando el dataset de *Prostate Cancer*:
1. Implementar un estimador de Ridge. Comparar con los resultados de la Tabla 3.3 del libro Elements of Statistical Learning. ¿Qué nivel de regularización se utilizó?¿Cómo decidirías el nivel de regularización?


In [2]:
# 2.1 — Implementar un estimador de Ridge
# TODO


---

### Ejercicio 3 — Dos atributos idénticos: cómo reparte Ridge

Una de las características de Ridge es que "reparte el peso" entre atributos correlacionados. Veámoslo en el caso extremo: supongamos que tenemos dos atributos idénticos, $x_2 = x_1$.

Mostrar que el término de error $|\textbf{y}-\textbf{X}\beta|^2$ depende de $\beta_1$ y $\beta_2$ solo a través de su suma $\beta_1+\beta_2$. ¿Qué implica esto sobre la unicidad de la solución de cuadrados mínimos?
Fijada esa suma en un valor $c$, minimizar el término de penalización $\beta_1^2+\beta_2^2$ sujeto a $\beta_1+\beta_2=c$. ¿Cuál es el reparto óptimo?


**Mi resolución:**

**(a) El error depende solo de $\beta_1 + \beta_2$.** Si $\mathbf{x}_2 = \mathbf{x}_1 =: \mathbf{x}$, la matriz de diseño es $\mathbf{X} = [\mathbf{x}\ \ \mathbf{x}]$ y

$$\mathbf{X}\beta = \beta_1\mathbf{x} + \beta_2\mathbf{x} = (\beta_1 + \beta_2)\,\mathbf{x}$$

Entonces

$$\|\mathbf{y} - \mathbf{X}\beta\|^2 = \|\mathbf{y} - (\beta_1+\beta_2)\,\mathbf{x}\|^2 =: g(\beta_1 + \beta_2)$$

es una función de la suma $s = \beta_1 + \beta_2$ únicamente: cualquier par con la misma suma produce **las mismas predicciones** y, por lo tanto, el mismo error.

**(b) Consecuencia: cuadrados mínimos no tiene solución única.** $g(s) = \|\mathbf{y}\|^2 - 2s\,\mathbf{x}^T\mathbf{y} + s^2\|\mathbf{x}\|^2$ es una parábola en $s$ con mínimo en $\hat s = \mathbf{x}^T\mathbf{y}/\|\mathbf{x}\|^2$ (regresión simple sobre $\mathbf{x}$). Pero todos los $\beta$ sobre la recta

$$\{(\beta_1, \beta_2) : \beta_1 + \beta_2 = \hat s\}$$

alcanzan ese mínimo: hay **infinitas soluciones**. En el paisaje de costo, el mínimo no es un punto sino un valle plano a lo largo de esa recta. Algebraicamente, $\mathbf{X}^T\mathbf{X} = \|\mathbf{x}\|^2\begin{pmatrix}1 & 1\\ 1 & 1\end{pmatrix}$ tiene rango 1 (determinante $0$), así que no es invertible y la fórmula $(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ no existe. Es el caso extremo de la multicolinealidad: el dato no tiene información para decidir cuánto del efecto es de $x_1$ y cuánto de $x_2$. Hasta $\hat\beta = (1000,\ \hat s - 1000)$ es "óptimo".

**(c) Reparto óptimo de la penalización.** Fijada la suma $\beta_1 + \beta_2 = c$, sustituimos $\beta_2 = c - \beta_1$:

$$h(\beta_1) = \beta_1^2 + (c - \beta_1)^2, \qquad h'(\beta_1) = 2\beta_1 - 2(c - \beta_1) = 4\beta_1 - 2c = 0 \ \Longrightarrow\ \beta_1 = \tfrac{c}{2}$$

y $h''(\beta_1) = 4 > 0$, así que es el mínimo. El reparto óptimo es **mitad y mitad**:

$$\beta_1 = \beta_2 = \frac{c}{2}, \qquad \beta_1^2 + \beta_2^2 = \frac{c^2}{2}$$

(Otra forma de verlo: $\beta_1^2 + \beta_2^2 = \tfrac{1}{2}(\beta_1+\beta_2)^2 + \tfrac{1}{2}(\beta_1-\beta_2)^2 = \tfrac{c^2}{2} + \tfrac{1}{2}(\beta_1-\beta_2)^2$, que se minimiza con $\beta_1 = \beta_2$.)

**(d) Qué hace Ridge entonces.** El costo de Ridge es $g(\beta_1+\beta_2) + \lambda(\beta_1^2+\beta_2^2)$. Para cualquier suma $s$, el primer término no distingue repartos y el segundo prefiere el reparto parejo; así que la solución de Ridge siempre cumple $\hat\beta_1 = \hat\beta_2$. Queda un problema en una sola variable, $\min_s\ g(s) + \lambda s^2/2$, cuya solución da

$$\hat\beta_1^{ridge} = \hat\beta_2^{ridge} = \frac{\mathbf{x}^T\mathbf{y}}{2\|\mathbf{x}\|^2 + \lambda}$$

(se verifica también resolviendo $(\mathbf{X}^T\mathbf{X} + \lambda I)\beta = \mathbf{X}^T\mathbf{y}$, que ahora sí es invertible). La penalización **rompe el empate**: de las infinitas soluciones de cuadrados mínimos elige una sola, y es la que reparte el peso por igual. Cuando $\lambda \to 0$ tiende a $\hat s/2$ para cada uno: la solución de cuadrados mínimos de **norma mínima**, la misma que da la pseudoinversa.


In [3]:
# Verificación numérica: dos columnas idénticas
rng = np.random.default_rng(0)
x = rng.normal(size=50)
y = 3 * x + rng.normal(scale=0.5, size=50)
X = np.column_stack([x, x])

# (a)-(b) El error solo depende de la suma: tres pares con suma s_hat dan el mismo RSS
s_hat = x @ y / (x @ x)
for b1 in [0.0, s_hat / 2, 1000.0]:
    beta = np.array([b1, s_hat - b1])
    print(f"beta = ({beta[0]:9.3f}, {beta[1]:9.3f})  ->  RSS = {np.sum((y - X @ beta) ** 2):.6f}")
print("rango de X^T X:", np.linalg.matrix_rank(X.T @ X))

# (d) Ridge reparte mitad y mitad
lam = 5.0
beta_ridge = np.linalg.solve(X.T @ X + lam * np.eye(2), X.T @ y)
print("\nRidge (lambda=5):", beta_ridge)
print("Fórmula x^T y / (2||x||^2 + lambda):", x @ y / (2 * x @ x + lam))
print("Pseudoinversa (lambda -> 0):", np.linalg.pinv(X) @ y, " vs  s_hat/2 =", s_hat / 2)


beta = (    0.000,     2.944)  ->  RSS = 12.589210
beta = (    1.472,     1.472)  ->  RSS = 12.589210
beta = ( 1000.000,  -997.056)  ->  RSS = 12.589210
rango de X^T X: 1

Ridge (lambda=5): [1.3899509 1.3899509]
Fórmula x^T y / (2||x||^2 + lambda): 1.3899508959676823
Pseudoinversa (lambda -> 0): [1.47202162 1.47202162]  vs  s_hat/2 = 1.4720216150620105
